# Demo: Orquestador Multi-Agente (Supervisor + Investigador + Analista)

Este notebook demuestra el flujo completo de delegación:

1. El usuario hace una consulta que requiere **investigación** y **análisis**.
2. El **Supervisor** decide enviarla primero al **Agente de Investigación**.
3. El Investigador busca en la base simulada de pre-entregas (`data/pre_entregas_kb.json`) y devuelve hallazgos.
4. El Supervisor recibe el resultado, decide que falta análisis y delega al **Agente de Análisis**.
5. El Analista calcula el promedio de calificaciones y el sentimiento general de los comentarios.
6. El Supervisor evalúa el resultado contra su rúbrica; si es insuficiente, pide **un** refinamiento; si ya está OK, **finaliza**.

> Requiere una `GOOGLE_API_KEY` (gratuita, https://aistudio.google.com/app/apikey) en un archivo `.env` en esta carpeta (ver `.env.example`).

In [1]:
from graph import app, route_from_supervisor
from langchain_core.messages import HumanMessage

## 1. Estructura del grafo (sin necesitar API key)

El grafo se puede inspeccionar y dibujar sin invocar ningún LLM, gracias a que los agentes se construyen de forma perezosa.

In [2]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	researcher(researcher)
	analyst(analyst)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analyst --> supervisor;
	researcher --> supervisor;
	supervisor -.-> __end__;
	supervisor -.-> analyst;
	supervisor -.-> researcher;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. Lógica anti-bucle-infinito (sin API key)

`decide_next_agent` es una función pura que garantiza que el grafo termina en como mucho `MAX_STEPS` pasos, sin importar lo que "opine" el LLM del Supervisor.

In [3]:
from agents.supervisor import decide_next_agent

# Simulación: el LLM siempre "quiere refinar", pero el tope de refinamientos manda.
has_research = has_analysis = False
sufficient = False
refinements_used = 0
steps = 0
trace = []

for _ in range(20):
    next_agent = decide_next_agent(has_research, has_analysis, sufficient, refinements_used, steps, llm_wants_refine=True)
    trace.append((steps, next_agent))
    if next_agent == "FINISH":
        break
    if next_agent == "researcher":
        has_research = True
    elif next_agent == "analyst":
        if has_analysis:
            refinements_used += 1
        has_analysis = True
    steps += 1

trace

[(0, 'researcher'), (1, 'analyst'), (2, 'analyst'), (3, 'FINISH')]

## 3. Ejecución real del flujo de delegación (requiere `GOOGLE_API_KEY` en `.env`)

In [4]:
query = (
    "Investigá qué feedback recibieron las pre-entregas del curso relacionadas "
    "con RAG y extracción de entidades, y luego analizá el sentimiento general "
    "de los comentarios y el promedio de las calificaciones."
)

initial_state = {
    "messages": [HumanMessage(content=query)],
    "user_query": query,
    "next_agent": "researcher",
    "task_completed": False,
    "research_data": "",
    "analysis_data": "",
    "contributions": [],
    "steps": 0,
    "refinements_used": 0,
}

final_state = app.invoke(initial_state)

C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [5]:
for c in final_state["contributions"]:
    print(f"[paso {c['step']}] {c['agent']}: {c['summary']}\n")

[paso 1] researcher: A continuación, presento los hallazgos encontrados en la base de datos de pre-entregas para la consulta sobre **RAG y extracción de entidades**:

1. **Pipeline de extracción de entidades**
   * **Cali

[paso 2] analyst: ### Resumen de Análisis de Pre-entregas

* **Promedio calculado:** 6.00 (Mínimo: 4.0, Máximo: 8.0)
* **Sentimiento general:** Negativo (5 señales negativas vs. 0 señales positivas en los comentarios)


[paso 3] analyst: A continuación, presento el resumen estructurado del análisis realizado sobre los datos de pre-entrega:

* **Promedio Calculado:** 6.00 (con un puntaje mínimo de 4.0 y un máximo de 8.0).
* **Sentimien



In [6]:
print("research_data:\n", final_state["research_data"])
print("\nanalysis_data:\n", final_state["analysis_data"])
print("\ntask_completed:", final_state["task_completed"])
print("refinements_used:", final_state["refinements_used"])

research_data:
 A continuación, presento los hallazgos encontrados en la base de datos de pre-entregas para la consulta sobre **RAG y extracción de entidades**:

1. **Pipeline de extracción de entidades**
   * **Calificación:** 8/10
   * **Comentario textual:** *"Buen uso de esquemas Pydantic para validar la salida del LLM. Faltó documentar los casos borde en el prompt de extracción."*

2. **Pipeline de extracción de entidades (schemas)**
   * **Calificación:** 4/10
   * **Comentario textual:** *"Faltaron validaciones críticas en el esquema de salida, generando errores silenciosos difíciles de detectar."*

analysis_data:
 A continuación, presento el resumen estructurado del análisis realizado sobre los datos de pre-entrega:

* **Promedio Calculado:** 6.00 (con un puntaje mínimo de 4.0 y un máximo de 8.0).
* **Sentimiento General:** Negativo (se identificaron 5 términos con carga negativa frente a 0 positivos en los comentarios).
* **Validación de Datos:** Válidos (contienen al menos un

## 4. También se puede ver paso a paso con `.stream()`

In [7]:
for chunk in app.stream(initial_state):
    for node, update in chunk.items():
        print(f"--- nodo: {node} ---")
        print({k: v for k, v in update.items() if k != "messages"})
        print()

--- nodo: supervisor ---
{'next_agent': 'researcher', 'task_completed': False}



C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- nodo: researcher ---
{'research_data': 'A continuación se presenta el resumen de los hallazgos obtenidos de la base de datos de pre-entregas para las consultas relacionadas con **RAG y extracción de entidades**:\n\n*   **Pre-entrega 1 (Pipeline de extracción de entidades)**\n    *   **Calificación:** 8/10\n    *   **Comentario textual:** "Buen uso de esquemas Pydantic para validar la salida del LLM. Faltó documentar los casos borde en el prompt de extracción."\n\n*   **Pre-entrega 1 (Pipeline de extracción de entidades - schemas)**\n    *   **Calificación:** 4/10\n    *   **Comentario textual:** "Faltaron validaciones críticas en el esquema de salida, generando errores silenciosos difíciles de detectar."', 'contributions': [{'agent': 'researcher', 'summary': 'A continuación se presenta el resumen de los hallazgos obtenidos de la base de datos de pre-entregas para las consultas relacionadas con **RAG y extracción de entidades**:\n\n*   **Pre-entrega 1 (Pipeli', 'step': 1}], 'steps':

C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


...

C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- nodo: analyst ---
{'analysis_data': 'A continuación, presento el resumen estructurado del análisis de las pre-entregas:\n\n*   **Promedio Calculado:** 6.0 (basado en las calificaciones 8/10 y 4/10).\n*   **Sentimiento General:** Negativo (con un balance de 0 palabras positivas frente a 5 negativas detectadas en los comentarios).\n*   **Validación de Datos:** Los datos son válidos y contienen la información mínima esperada (calificaciones numéricas y texto libre suficiente).', 'contributions': [{'agent': 'analyst', 'summary': 'A continuación, presento el resumen estructurado del análisis de las pre-entregas:\n\n*   **Promedio Calculado:** 6.0 (basado en las calificaciones 8/10 y 4/10).\n*   **Sentimiento General:** Negativo (c', 'step': 3}], 'steps': 3}



C:\Users\fluca\OneDrive\Escritorio\Pre_Entrega6\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- nodo: supervisor ---
{'next_agent': 'FINISH', 'task_completed': True}

